# Using LLMS 

In [1]:
# General
import os
from pathlib import Path
from tqdm import tqdm

# data manipulation
import json
import pandas as pd
# import numpy as np

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
# from matplotlib import cm

# Interactive Jupyter
from IPython.display import display

# others
import contractions
from scipy.stats import pearsonr, ttest_ind 
import nltk
nltk.download('stopwords')
import random

# LLMs
import ollama

os.chdir('../Milestone_P2')
# Custom utility python files
import utils.visualization as viz # custom visualization utils in '/utils/visualization.py'
import utils.visualization2 as viz2 # custom visualization utils in '/utils/visualization2.py'
import utils.data_wrangling as dw # custom data wrangling utils in '/utils/data_wrangling.py'
import utils.structural_analysis as ut # custom structural analysis utils in '/utils/structural_analysis.py'
# import utils.data_augmentation as da # custom data augmentation utils in '/utils/data_augmentations.py'

os.chdir('../Adrian')

PROJECT_ROOT = Path.cwd().parent.parent
FIGURES_DIR = PROJECT_ROOT / 'figures'

# DATA DIRECTORIES
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'cleaned_data'
IMG_DIR = DATA_DIR / 'images'

print("Project path setup:")
print("PROJECT_ROOT:  ", PROJECT_ROOT)
print("FIGURES_DIR:   ", FIGURES_DIR)
print("\nDATA_DIR:      ", DATA_DIR)
print("JSON_DIR:      ", JSON_DIR)
print("CSV_DIR:       ", CSV_DIR)
print("CSV_CLEAN_DIR: ", CSV_CLEAN_DIR)
print("IMG_DIR:       ", IMG_DIR)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adrian/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Project path setup:
PROJECT_ROOT:   /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo
FIGURES_DIR:    /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/figures

DATA_DIR:       /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/data/newyorker_caption_contest
JSON_DIR:       /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/data/newyorker_caption_contest/contests.json
CSV_DIR:        /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/data/newyorker_caption_contest/data
CSV_CLEAN_DIR:  /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/data/newyorker_caption_contest/cleaned_data
IMG_DIR:        /Users/adrian/Desktop/Etudes/EPFL/2025-2026/MA1/ADA/ada-2025-project-json-derulo/data/newyorker_caption_contest/images


CITE THE DATASET !!!!!
If you use this dataset, we would appreciate you citing our work, but also -- several other papers that we build this corpus upon. See Citation Information.

## Image description

In [5]:
import base64
def _missing_meta(meta, what):
    """
    Check if a single metadata dictionary is missing a given key.
    meta: dictionary that is supposed to contain all the metadata
    what: key that we are looking for
    """
    return (
        not isinstance(meta, dict)
        or what not in meta
        )
    

def ids_missing_key(df, what):
    """
    Returns the contest IDs (not indices) for which the metadata dictionary is missing the key `what`.

    Args:
        metadata_list (list): List of metadata dicts (contests_df['metadata'])
        what (str): Key to check for (e.g. 'num_captions', 'image_descriptions', etc.)
        contest_ids (list or Series): List of contest IDs aligned with metadata_list
    """
    return [
        int(df['contest_id'][i])
        for i, meta in enumerate(df['metadata'])
        if _missing_meta(meta, what)
    ]
    
def _examples_for_ai(df,root,number_of_ex=5):
    examples = []
    for _, complete in random.sample(list(df.iterrows()),number_of_ex):
        image_path=root+'/'+str(complete['contest_id'])+'.jpg'
        with open(image_path, "rb") as f:
                #image = f.read() #test 1
                #image =base64.b64encode(f.read()).decode("utf-8") # test2 
                image=image_path #test3
        meta = complete['metadata']
        
        examples.append({ 
            'image_path': image, 'image_loc': meta.get('image_locations'),
            'image_descri': meta.get('image_descriptions'), 'image_uncan_descri':meta.get('image_uncanny_descriptions'),
            'images_entities': meta.get('entities'),'images_questions': meta.get('questions')
        })
    return examples

def _prompt_making_description(examples):
    prompt = "You are an expert that generates detailed and structured metadata for images.\n\n"
    prompt+=f'Next are some examples for different images, I want your answer to be in the same structure as in the examples\n'    
    for example in examples:  # Fewer examples for smaller models
        prompt += f"Example:\n"
        #prompt += f"image_path: {example['image_path']}\n" # or image_path
        prompt += f"Location: {example['image_loc']}\n"
        prompt += f"Description: {example['image_descri']}\n"
        prompt += f"Uncanny description : {example['image_uncan_descri']}\n"
        prompt += f"Entities: {example['images_entities']}\n"
        prompt += f"Questions: {example['images_questions']}\n\n"
    prompt+='Now to our image !'
    return prompt

def add_image_descriptions(contests_df, root=IMG_DIR): 
    '''
    contests_df: contests dataframe from the Json 
    
    root: Path or str that points the folder where the images are
    ''' 
    root=str(root)
    # Test if Ollama is running and pull a model
    try:
        ollama.list()
    except:
        print("Please install and start Ollama first: https://ollama.ai/")
        return contests_df
    missing_df_ids = ids_missing_key(contests_df,'image_descriptions')
    complete_df = contests_df[~contests_df['contest_id'].isin(missing_df_ids)] ## removes the missings dfs

    required_keys = [
        "image_locations : what location is the image decripting", "image_descriptions : What is happening in the image", "image_uncanny_descriptions : what is special about the image situation",
        "entities: wikipedias links to the entities from the image", "questions: what questions can be asked from the image"," your answer confidence: (between 0-1)"
    ]
    results = []
    for ids in tqdm(missing_df_ids):
        
        meta = contests_df.loc[contests_df['contest_id'] == ids, 'metadata'].iloc[0]
        if not isinstance(meta, dict):
            meta = {}
        image_path=root+'/'+str(ids)+'.jpg'
        examples = _examples_for_ai(complete_df,root,number_of_ex=3)    
        prompt=_prompt_making_description(examples)
        with open(image_path, "rb") as f:
                #image = f.read() test 1
                image =base64.b64encode(f.read()).decode("utf-8") #test 2
                #image=image_path #test3
        #prompt = "You are an expert that generates detailed and structured metadata for images.\n\n"
        prompt += f"""
        Fill in the missing keys: {', '.join(required_keys)}, for the image.
        """
        print(f'id : {ids}, prompt {prompt}')
        try:
            response = ollama.generate(
                model='bakllava:7b',  # 'llava:7b', 'bakllava:7b'
                prompt=prompt,
                images=[image],
                options={'temperature': 0.1}  # control randomness/creativity, low temp mostly consistent, slight creativity
            )
            description = response['response']
        
            results.append({
            'contest_id': ids,
            'description': description
            })
            
            print(results)
        except Exception as e:
            print(f"Error processing contest {ids}: {e}")
        break
    return results


In [6]:
file_path = str(JSON_DIR)
with open(file_path, "r") as file:
    contests = json.load(file)
contests_df = pd.DataFrame(contests)
results=add_image_descriptions(contests_df)

  0%|          | 0/144 [00:00<?, ?it/s]

id : 527, prompt You are an expert that generates detailed and structured metadata for images.

Next are some examples for different images, I want your answer to be in the same structure as in the examples
Example:
Location: ['airplane']
Description: ["A large group of diverse people are sitting in the middle aisle seats of a plain. A blonde woman with a laptop on her lap is looking to her left at a man wearing a chef's hat and uniform, throwing pizza dough into the air.", 'This is a plane full of people. A baker is tossing a pizza on the plane. The woman next to him looks puzzled.', "A number of people are seated in a plane. A woman is having a conversation with a man wearing a chef's outfit."]
Uncanny description : ["A chef wouldn't be wearing his work clothes onto a plane.", "It's unusual to see someone kneading pizza dough, throwing it up in the air, on an airplane.  TSA would likely have confiscated their dough.", 'There is a baker on the plane tossing a pizza.']
Entities: ['http

  0%|          | 0/144 [05:05<?, ?it/s]

[{'contest_id': 527, 'description': '0'}]


In [4]:
print(results)

[{'contest_id': 527, 'description': '0'}]


## API ?

In [ ]:
import os
from ollama import Client

client = Client(
    host='https://ollama.com',
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

messages = [
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
]

for part in client.chat('gpt-oss:120b', messages=messages, stream=True):
  print(part.message.content, end='', flush=True)

## Joke discription

In [ ]:
# Caption only

Cat1=['Dark Humour','Irony','Social Commentary','Wordplay','Absurdity']
Cat2=['Language','Logic','Identity','Action'] # Berger’s humor categories

def _prompt_making_classifictation(cat,description=[],image=False):
    prompt=f"You are an expert in joke classification !\n"
    prompt+=f"Your job is to classify each joke to the following categories : {', '.join(cat)}.\n"
    prompt+=f"I want you to assert the confidence with which you're attributing the category. For example : 0.3 Langauage, 0.6 Logic, 0.0 Identity, 0.1 Action\n"
    prompt+=f"The definitions of the categories are :\n"
    promtp+=f"1) Language; Verbal humor, Allusion, Irony, Puns, Repartee, Wit, Ridicule\n"
    prompt+=f"2) Logic; Ideational humor, Absurdity, Coincidence, Conceptual surprise, Outwitting, Caught out, Misunderstanding\n"
    prompt+=f"3) Identity; Existential humor, Parody, Rigidity, Malicious pleasure, Condescension, Deceitful behavior, Self-deprecation\n"
    prompt+=f"4) Action; Physical/ non-verbal humour, Peculiar face, Peculiar music, Clumsiness, Repulsive behavior\n"
    if description:
        prompt+=f"Use the image description as well as the uncanny image discription to assert the caption category :\n "
        prompt+=f"Description; {description[0]}\n"
        prompt+=f"Uncanny description; {description[1]}\n"
    elif image:
        prompt+=f"Use the image to assert the caption category\n"
    return prompt

def classify_captions(cat,image_description=False,image=False):
    '''
    Takes one contest, df, and attribute the captions into one from the given categories, cat.
    It does so using ollama with a specific model. 
    It returns df with a new column with the joke classification
    '''
    file_path = str(JSON_DIR)
    with open(file_path, "r") as file:
        contests = json.load(file)
    contests_df = pd.DataFrame(contests)
        
    for contest in os.listdir(str(CSV_DIR)):
        contest_id=contest.split('/')[-1].removesuffix('.csv')
        df=pd.read_csv(contest)
        descrip=[]
        if image_description:
            contest=contests_df[[contests_df['contest_id']==contest_id]]
            descrip=[contest['metadata'].get('image_descriptions'),contest['metadata'].get('image_uncanny_descriptions')]
        prompt=_prompt_making_classifictation(cat,descrip)
        results = {}
        for idx, row in df.iterrows():
            prompt_cap=prompt + f'Classify this caption : ' + row['caption']
            try:
                response = ollama.generate(
                    model='llava-llama3:8b',  # 'llava:7b', 'llava-llama3:8b'
                    prompt=prompt_cap,
                    #images=[image],
                    options={'temperature': 0.1}  # control randomness/creativity, low temp mostly consistent, slight creativity
                )
                description = response['response']
                results[str(idx)]=description
                
            except Exception as e:
                print(f"Error processing contest {idx}: {e}")
                continue
            break
    return results

print("JSON_DIR:      ", JSON_DIR)
print("CSV_DIR:       ", CSV_DIR)
print("CSV_CLEAN_DIR: ", CSV_CLEAN_DIR)
print("IMG_DIR:       ", IMG_DIR)
    



